# Pyspark Jamming 2024/06/15

In [1]:
import findspark
findspark.init()
findspark.find()
import pyspark
findspark.find()

'C:\\Users\\micha\\AppData\\Local\\spark\\spark-3.4.0-bin-hadoop3'

In [49]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import BooleanType

In [3]:
spark = (
    SparkSession.builder.appName('ReadALoadOfFiles')
    .config('spark.executro.instances', '10')
    .config("spark.executor.memory", "6g")
    .config("spark.driver.memory", "6g")
    .config("spark.jars.packages", "com.crealytics:spark-excel_2.12:0.13.5")
    .getOrCreate() 
    )

In [4]:
# import python packages
import os
import zipfile
import xml.etree.ElementTree as ET

## Read Excel Files

In [5]:
path_downloads = r'C:\Users\micha\Downloads'
file_ext = '.xlsx'
list_files = [x for x in os.listdir(path_downloads) if os.path.splitext(x)[1] == file_ext]
list_files

['online_retail_II.xlsx']

In [6]:
def get_excel_sheet_names(file_path):
    # Open the Excel file as a zip archive
    with zipfile.ZipFile(file_path, 'r') as z:
        # Read the workbook.xml file
        with z.open('xl/workbook.xml') as f:
            tree = ET.parse(f)
            root = tree.getroot()
            # Define the namespaces
            namespaces = {'main': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
            # Extract sheet names
            sheet_names = []
            for sheet in root.findall('main:sheets/main:sheet', namespaces):
                sheet_names.append(sheet.get('name'))
    
    return sheet_names

In [7]:
dict_excel_sheets = {}

for file in list_files:
    path_file = os.path.join(path_downloads, file)
    sheet_names = get_excel_sheet_names(path_file)
    dict_excel_sheets[path_file] = sheet_names


In [8]:
dict_excel_sheets

{'C:\\Users\\micha\\Downloads\\online_retail_II.xlsx': ['Year 2009-2010',
  'Year 2010-2011']}

In [9]:
dict_dfs = {}
for xls, sheets in dict_excel_sheets.items():
    for sheet in sheets:
        print(f'{xls} : {sheet}')
        df = (spark.read
            .format("com.crealytics.spark.excel")
            .option("maxRowsInMemory", 100000) 
            .option('header', 'true')
            .option("samplingRatio", "0.0001")
            .option('inferSchema', 'true')
            .option('dataAddress', f"'{sheet}'!A1")
            # .option("startRow", 1)
            # .option("endRow", 10)
            .load(xls)
            )        
        
        dict_dfs[xls] = {sheet : df}

C:\Users\micha\Downloads\online_retail_II.xlsx : Year 2009-2010
C:\Users\micha\Downloads\online_retail_II.xlsx : Year 2010-2011


In [10]:
df_1 = dict_dfs['C:\\Users\\micha\\Downloads\\online_retail_II.xlsx']['Year 2010-2011']

In [11]:
type(df_1)

pyspark.sql.dataframe.DataFrame

## Analysis

In [12]:
df_1.count()

541910

In [13]:
df_1.printSchema()

root
 |-- Invoice: double (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)



In [14]:
# df_1.columns
df_1.select('Description').distinct().count()

4224

In [15]:
df_1.describe().show()

+-------+--------+------------------+--------------------+-----------------+-----------------+------------------+-----------+
|summary| Invoice|         StockCode|         Description|         Quantity|            Price|       Customer ID|    Country|
+-------+--------+------------------+--------------------+-----------------+-----------------+------------------+-----------+
|  count|  541910|            541910|              540456|           541910|           541910|            406830|     541910|
|   mean|     NaN|27623.240210938104|             20713.0|9.552233765754462|4.611138332934989|15287.684160460143|       null|
| stddev|     NaN|16799.737628427687|                null|218.0809569439247|96.75976549366524|1713.6030743514796|       null|
|    min|536365.0|             10002| 4 PURPLE FLOCK D...|         -80995.0|        -11062.06|           12346.0|  Australia|
|    max|     NaN|                 m|   wrongly sold sets|          80995.0|          38970.0|           18287.0|Unspe

In [16]:
clmn = df_1.columns[1]
print(clmn)

# df_1.filter(df_1[clmn] > 500000).createOrReplaceTempView('df_2')
# spark.sql('select * from df_2').show() # works count does not
# spark.catalog.dropTempView("df_2")

df_2 = df_1.filter(df_1[clmn] > '500000')
df_2.show()

StockCode
+--------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
| Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|
+--------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
|536365.0|   85123A|WHITE HANGING HEA...|     6.0|2010-12-01 08:26:00| 2.55|    17850.0|United Kingdom|
|536365.0|    71053| WHITE METAL LANTERN|     6.0|2010-12-01 08:26:00| 3.39|    17850.0|United Kingdom|
|536365.0|   84406B|CREAM CUPID HEART...|     8.0|2010-12-01 08:26:00| 2.75|    17850.0|United Kingdom|
|536365.0|   84029G|KNITTED UNION FLA...|     6.0|2010-12-01 08:26:00| 3.39|    17850.0|United Kingdom|
|536365.0|   84029E|RED WOOLLY HOTTIE...|     6.0|2010-12-01 08:26:00| 3.39|    17850.0|United Kingdom|
|536367.0|    84879|ASSORTED COLOUR B...|    32.0|2010-12-01 08:34:00| 1.69|    13047.0|United Kingdom|
|536367.0|    84969|BOX OF 6 ASSORTED...|     6.0|2010

In [17]:
df_2.count()

78617

In [18]:
df_3 = df_2.repartition(10)

In [19]:
df_2.dtypes

[('Invoice', 'double'),
 ('StockCode', 'string'),
 ('Description', 'string'),
 ('Quantity', 'double'),
 ('InvoiceDate', 'timestamp'),
 ('Price', 'double'),
 ('Customer ID', 'double'),
 ('Country', 'string')]

In [21]:
df_3 = (df_3
        .withColumn('Invoice', col('Invoice').cast('string'))
        .withColumn('Customer ID', col('Customer ID').cast('string'))
    )

In [22]:
for x in df_3.schema:
    print(x)

StructField('Invoice', StringType(), True)
StructField('StockCode', StringType(), True)
StructField('Description', StringType(), True)
StructField('Quantity', DoubleType(), True)
StructField('InvoiceDate', TimestampType(), True)
StructField('Price', DoubleType(), True)
StructField('Customer ID', StringType(), True)
StructField('Country', StringType(), True)


In [27]:
df_3.persist()

DataFrame[Invoice: string, StockCode: string, Description: string, Quantity: double, InvoiceDate: timestamp, Price: double, Customer ID: string, Country: string]

In [32]:
df_3.filter(col('Quantity').contains(1)).show(5)

+--------+---------+--------------------+--------+-------------------+------+-----------+--------------+
| Invoice|StockCode|         Description|Quantity|        InvoiceDate| Price|Customer ID|       Country|
+--------+---------+--------------------+--------+-------------------+------+-----------+--------------+
|543235.0|    71459|HANGING JAM JAR T...|    12.0|2011-02-04 13:30:00|  0.85|    14462.0|United Kingdom|
|543345.0|      DOT|      DOTCOM POSTAGE|     1.0|2011-02-07 12:29:00|183.09|       null|United Kingdom|
|541423.0|    70006|LOVE HEART POCKET...|     1.0|2011-01-17 17:54:00|  1.63|       null|United Kingdom|
|539958.0|   84251C|GREETING CARD, TW...|    13.0|2010-12-23 13:26:00|  0.43|       null|United Kingdom|
|540995.0|   85071D|CHARLIE+LOLA MY R...|     1.0|2011-01-13 09:30:00|  0.83|       null|United Kingdom|
+--------+---------+--------------------+--------+-------------------+------+-----------+--------------+
only showing top 5 rows



In [47]:
## value counts()
(
df_3.select(
        col('Quantity')
        .contains(1)
         )
    .groupby('contains(Quantity, 1)')
    .count()
    .show()
    )

+---------------------+-----+
|contains(Quantity, 1)|count|
+---------------------+-----+
|                 true|37584|
|                false|41033|
+---------------------+-----+



In [64]:
(
df_3.select(col('Customer ID').isNull()).show(5)
)

+---------------------+
|(Customer ID IS NULL)|
+---------------------+
|                false|
|                false|
|                 true|
|                false|
|                false|
+---------------------+
only showing top 5 rows

